<a href="https://colab.research.google.com/github/lakshaykumar11/tts-quantization-indic/blob/main/tts_quant_fp32_local.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q coqui-tts
!pip install -q -U "transformers<5"
!pip install -q -U openai-whisper
!pip install -q jiwer indic-transliteration

!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


In [2]:
import os

if not os.path.exists('/content/.restarted'):
    open('/content/.restarted', 'w').close()
    os.kill(os.getpid(), 9)
print("ready")

ready


In [3]:
import os, re, time, wave, torch
import pandas as pd, jiwer
from TTS.api import TTS
import whisper
from google.colab import files
from indic_transliteration import sanscript
from indic_transliteration.sanscript import transliterate

OUT = '/content/tts_quant'
os.makedirs(f'{OUT}/audio', exist_ok=True)
os.makedirs(f'{OUT}/results', exist_ok=True)

PRECISION = 'fp32'
SPEAKER = 'Ana Florence'
TAG = {'EN': 'en', 'HI': 'hi', 'HING_MIX': 'hi', 'HING_ROM': 'en'}
ASR_LANG = {'EN': 'en', 'HI': 'hi', 'HING_MIX': 'hi', 'HING_ROM': 'hi'}
NEUTRAL = {'EN': False, 'HI': False, 'HING_MIX': True, 'HING_ROM': True}

tts = TTS('tts_models/multilingual/multi-dataset/xtts_v2')
asr = whisper.load_model('large-v3')
print('models loaded ->', OUT)

 > You must confirm the following:
 | > "I have purchased a commercial license from Coqui: licensing@coqui.ai"
 | > "Otherwise, I agree to the terms of the non-commercial CPML: https://coqui.ai/cpml" - [y/n]
 | | > y


100%|██████████| 1.87G/1.87G [00:41<00:00, 45.4MiB/s]
4.37kiB [00:00, 1.87MiB/s]
361kiB [00:00, 87.4MiB/s]
100%|██████████| 32.0/32.0 [00:00<00:00, 95.3kiB/s]
100%|██████████| 7.75M/7.75M [00:00<00:00, 16.0MiB/s]
100%|█████████████████████████████████████| 2.88G/2.88G [00:31<00:00, 97.8MiB/s]


models loaded -> /content/tts_quant


In [4]:
DEVA = re.compile(r'[\u0900-\u097F]')
CONS = 'bcdfghjklmnpqrstvwxyz'


def _tokens(text):
    out = []
    for tok in text.split():
        if DEVA.search(tok):
            out.append((transliterate(tok, sanscript.DEVANAGARI, sanscript.ITRANS), True))
        else:
            out.append((tok, False))
    return out


def _norm_word(w, from_deva):
    if from_deva:
        w = w.replace('A', 'aa').replace('I', 'ii').replace('U', 'uu')
        w = w.replace('M', 'n').replace('~', 'n').replace('.', '')
    w = re.sub(r'[^a-z]', '', w.lower())
    if not w:
        return ''
    if from_deva:
        w = re.sub(r'(?<!a)a$', '', w)
        prev = None
        while prev != w:
            prev = w
            w = re.sub(r'([aeiou][%s])a([%s][aeiou])' % (CONS, CONS), r'\1\2', w)
    w = w.replace('w', 'v').replace('y', 'i')
    w = w.replace('aa', 'a').replace('ii', 'i').replace('uu', 'u')
    w = w.replace('ee', 'i').replace('oo', 'u')
    w = w.replace('ai', 'e').replace('au', 'o')
    w = w.replace('c', 'k').replace('q', 'k').replace('x', 'ks')
    w = re.sub(r'([%s])h' % CONS, r'\1', w)
    return re.sub(r'(.)\1+', r'\1', w)


def phonetic_norm(text):
    return ' '.join(x for x in (_norm_word(t, d) for t, d in _tokens(text)) if x)


def plain_norm(text):
    text = re.sub(r"[।.,!?;:\"'\-]", '', text.lower())
    return re.sub(r'\s+', ' ', text).strip()


def wav_seconds(path):
    with wave.open(path) as f:
        return f.getnframes() / f.getframerate()

In [5]:
import shutil

# If resuming: upload the last downloaded fp32.csv, then run this.
if os.path.exists('fp32.csv'):
    shutil.copy('fp32.csv', f'{OUT}/results/fp32.csv')
    print('checkpoint restored')
else:
    print('no checkpoint to restore - starting fresh')


no checkpoint to restore - starting fresh


In [6]:
df = pd.read_csv('testset.csv')
RESULTS = f'{OUT}/results/{PRECISION}.csv'

if os.path.exists(RESULTS):
    done = pd.read_csv(RESULTS)
    seen = set(done['id'])
else:
    done, seen = pd.DataFrame(), set()

todo = df[~df['id'].isin(seen)]
print(f'{len(seen)} done, {len(todo)} remaining')

0 done, 300 remaining


In [7]:
rows, t_start = [], time.time()

for n, (_, r) in enumerate(todo.iterrows(), 1):
    cond = r.condition
    path = f'{OUT}/audio/{PRECISION}_{r.id}.wav'

    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    tts.tts_to_file(text=r.text, speaker=SPEAKER,
                    language=TAG[cond], file_path=path)
    synth_s = time.time() - t0
    peak_mb = torch.cuda.max_memory_allocated() / 1024**2

    heard = asr.transcribe(path, language=ASR_LANG[cond])['text']
    f = phonetic_norm if NEUTRAL[cond] else plain_norm
    ref, hyp = f(r.text), f(heard)
    audio_s = wav_seconds(path)

    rows.append({
        'id': r.id, 'precision': PRECISION, 'condition': cond,
        'mix_level': r.mix_level, 'cmi': r.cmi, 'n_words': r.n_words,
        'tts_lang': TAG[cond], 'wer': jiwer.wer(ref, hyp), 'cer': jiwer.cer(ref, hyp),
        'synth_s': round(synth_s, 2), 'audio_s': round(audio_s, 2),
        'rtf': round(synth_s / audio_s, 3), 'peak_mb': round(peak_mb, 1),
        'text': r.text, 'heard': heard.strip(),
    })

    if n % 10 == 0 or n == len(todo):
        pd.concat([done, pd.DataFrame(rows)]).to_csv(RESULTS, index=False)
        if n % 50 == 0:
            files.download(RESULTS)
        el = time.time() - t_start
        eta = el / n * (len(todo) - n) / 60
        print(f'{n}/{len(todo)}  elapsed {el/60:.0f}m  eta {eta:.0f}m')

print('done')

10/300  elapsed 5m  eta 135m
20/300  elapsed 8m  eta 115m
30/300  elapsed 12m  eta 107m
40/300  elapsed 16m  eta 102m


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

50/300  elapsed 19m  eta 96m
60/300  elapsed 23m  eta 92m
70/300  elapsed 27m  eta 87m
80/300  elapsed 30m  eta 83m
90/300  elapsed 34m  eta 79m


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

250/300  elapsed 113m  eta 23m
260/300  elapsed 119m  eta 18m
270/300  elapsed 124m  eta 14m
280/300  elapsed 130m  eta 9m
290/300  elapsed 135m  eta 5m


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

300/300  elapsed 140m  eta 0m
done


In [8]:
res = pd.read_csv(RESULTS)
print(len(res), 'rows\n')
print(res.groupby('condition')[['wer', 'cer', 'rtf']].mean().round(3), '\n')
print(res[res.condition.isin(['HING_MIX', 'HING_ROM'])]
      .groupby(['condition', 'mix_level'])[['cmi', 'wer', 'cer']].mean().round(3))

300 rows

             wer    cer    rtf
condition                     
EN         0.004  0.004  4.649
HI         0.229  0.096  4.593
HING_MIX   0.382  0.213  4.575
HING_ROM   0.595  0.198  4.570 

                        cmi    wer    cer
condition mix_level                      
HING_MIX  high       40.440  0.419  0.194
          low        10.880  0.301  0.164
          medium     24.497  0.432  0.277
HING_ROM  high       40.440  0.606  0.195
          low        10.880  0.594  0.207
          medium     24.497  0.585  0.191


In [9]:
h = res[res.condition.isin(['HING_MIX', 'HING_ROM'])]
for c in ['HING_MIX', 'HING_ROM']:
    s = h[h.condition == c]
    print(f"{c}: CMI-WER r = {s['cmi'].corr(s['wer']):.3f}  (n={len(s)})")

HING_MIX: CMI-WER r = 0.156  (n=100)
HING_ROM: CMI-WER r = 0.030  (n=100)
